
# Cloud-First Arabic RAG on Google Colab — Fixed Version

هذه النسخة مصممة بحيث يكون Colab مجرد **Orchestrator خفيف**، بينما المعالجة الثقيلة تتم عبر خدمات سحابية.

## Architecture

```text
PDF
 │
 ▼
LlamaParse Cloud
 │
 ▼
LlamaIndex
 │
 └── Sentence-aware Chunking
 │
 ▼
Jina Embeddings API
 │
 ▼
Qdrant Cloud
 ├── Dense vectors
 └── BM25 sparse vectors
 │
 ▼
Hybrid Search
Dense + BM25 + RRF
 │
 ▼
Jina Reranker API
 │
 ▼
Top-N Context
 │
 ▼
Hugging Face Inference API
 │
 ▼
Arabic Answer + Citations
```

## أهم التحسينات في هذه النسخة

- إصلاح ترتيب الـpipeline بالكامل.
- رفع الـChunks فعليًا إلى Qdrant بعد إنشاء الـCollection.
- عدم حذف الـCollection تلقائيًا إلا عند اختيار ذلك صراحةً.
- معالجة Jina rate limits بالـbatching والـretry.
- فحص عدد الـPoints داخل Qdrant بعد الـupload.
- Debug واضح لكل مرحلة.
- واجهة سؤال بمربع نص وزر بدل تعديل الكود يدويًا.
- إمكانية طرح عدة أسئلة دون إعادة الـingestion.


In [ ]:

# ============================================================
# 1) تثبيت المكتبات
# ============================================================

!pip install -q -U \
    "llama-cloud>=2.8" \
    llama-index-core \
    llama-index-embeddings-jinaai \
    llama-index-postprocessor-jinaai-rerank \
    "qdrant-client>=1.15.2" \
    openai \
    ipywidgets


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.6/481.6 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.6/164.6 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/14


## 2. مفاتيح الخدمات

ستحتاج:

- `LLAMA_CLOUD_API_KEY`
- `JINAAI_API_KEY`
- `QDRANT_URL`
- `QDRANT_API_KEY`
- `HF_TOKEN`


In [ ]:

# ============================================================
# 2) تحميل مفاتيح الخدمات
# ============================================================

import os
from getpass import getpass

os.environ["LLAMA_CLOUD_API_KEY"] = getpass("LLAMA_CLOUD_API_KEY: ")
os.environ["JINAAI_API_KEY"] = getpass("JINAAI_API_KEY: ")

qdrant_url = input("QDRANT_URL: ").strip()
os.environ["QDRANT_URL"] = qdrant_url

os.environ["QDRANT_API_KEY"] = getpass("QDRANT_DATABASE_API_KEY: ")
os.environ["HF_TOKEN"] = getpass("HF_TOKEN: ")

print("✅ تم تحميل مفاتيح الخدمات")


LLAMA_CLOUD_API_KEY: ··········
JINAAI_API_KEY: ··········
QDRANT_URL: https://a1200afa-0cd2-416d-9a20-70edd4ed6d28.sa-east-1-0.aws.cloud.qdrant.io
QDRANT_DATABASE_API_KEY: ··········
HF_TOKEN: ··········
✅ تم تحميل مفاتيح الخدمات


In [ ]:

# ============================================================
# 3) الإعدادات العامة
# ============================================================

COLLECTION_NAME = "cloud_rag_demo"

# Cloud embedding
EMBED_MODEL_NAME = "jina-embeddings-v3"
EMBED_DIM = 1024

# Cloud reranker
RERANK_MODEL_NAME = "jina-reranker-v2-base-multilingual"

# Hugging Face LLM
LLM_MODEL_NAME = "Qwen/Qwen3.5-9B"

# Chunking
CHUNK_SIZE = 800
CHUNK_OVERLAP = 80

# Retrieval
DENSE_CANDIDATES = 12
SPARSE_CANDIDATES = 12
RRF_TOP_K = 12
RERANK_TOP_N = 5

# Jina rate-limit handling
EMBED_BATCH_SIZE = 6
WAIT_BETWEEN_BATCHES = 3
RATE_LIMIT_RETRY_WAIT = 30
MAX_RETRIES = 5

print("Embedding model:", EMBED_MODEL_NAME)
print("Embedding dimension:", EMBED_DIM)
print("Reranker:", RERANK_MODEL_NAME)
print("LLM:", LLM_MODEL_NAME)


Embedding model: jina-embeddings-v3
Embedding dimension: 1024
Reranker: jina-reranker-v2-base-multilingual
LLM: Qwen/Qwen3.5-9B


## 4. رفع ملف PDF

In [ ]:

# ============================================================
# 4) رفع ملف PDF
# ============================================================

from google.colab import files

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("❌ لم يتم رفع أي ملف.")

PDF_PATH = next(iter(uploaded.keys()))

print("✅ الملف:", PDF_PATH)


Saving رسالة كندة ابراهيم زيدان.pdf to رسالة كندة ابراهيم زيدان.pdf
✅ الملف: رسالة كندة ابراهيم زيدان.pdf



## 5. Parsing سحابي عبر LlamaParse

هذه المرحلة ترسل الملف إلى LlamaParse لاستخراج Markdown من الصفحات، بما في ذلك OCR والجداول.


In [ ]:

# ============================================================
# 5) LlamaParse Cloud
# ============================================================

from llama_cloud import LlamaCloud

llama_client = LlamaCloud()

print("⏳ رفع الملف إلى LlamaCloud...")

cloud_file = llama_client.files.create(
    file=PDF_PATH,
    purpose="parse",
)

print("✅ File ID:", cloud_file.id)

print("⏳ Parsing...")

parse_result = llama_client.parsing.parse(
    file_id=cloud_file.id,
    tier="agentic",
    version="latest",
    output_options={
        "markdown": {
            "tables": {
                "output_tables_as_markdown": True
            }
        },
        "images_to_save": ["screenshot"],
    },
    processing_options={
        "ocr_parameters": {
            "languages": ["ar", "en"]
        }
    },
    expand=["markdown"],
)

pages = parse_result.markdown.pages

if not pages:
    raise RuntimeError("❌ LlamaParse لم يُرجع صفحات.")

print(f"✅ عدد الصفحات المستخرجة: {len(pages)}")

print("\n--- مثال من أول صفحة ---\n")
print((pages[0].markdown or "")[:2500])


⏳ رفع الملف إلى LlamaCloud...
✅ File ID: 88c6ddf6-9420-4a5f-90b5-9e944c3ab95f
⏳ Parsing...
✅ عدد الصفحات المستخرجة: 138

--- مثال من أول صفحة ---

Damascus University logo

الجمهورية العربية السورية
جامعة دمشق
كلية الهندسة الزراعية
قسم الاقتصاد الزراعي

# دور سياسة التسويق الزراعي في تحسين كفاءة أداء أسواق الجملة لتجارة أهم منتجات الخضار والفاكهة في محافظة حماة

رسالة أُعدّت لنيل درجة الماجستير في الهندسة الزراعية
(اختصاص - اقتصاد زراعي)

**إعداد الطالبة**
**كنده إبراهيم زيدان**

**المشرف**
**الدكتور شباب ناصر**
أستاذ في قسم الاقتصاد الزراعي

**المشرف العلمي المشارك**
**الدكتور فادي مقدسي**
مدرس في قسم الاقتصاد الزراعي

العام الدراسي 1445هـ/2024م


## 6. تحويل الصفحات إلى Documents وChunks عبر LlamaIndex

In [ ]:

# ============================================================
# 6) Documents + Chunking
# ============================================================

from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter

documents = []

for page_number, page in enumerate(pages, start=1):
    text = (page.markdown or "").strip()

    if not text:
        continue

    documents.append(
        Document(
            text=text,
            metadata={
                "source": PDF_PATH,
                "page": page_number,
            },
        )
    )

if not documents:
    raise RuntimeError("❌ لا يوجد نص مستخرج صالح للتحويل إلى Documents.")

splitter = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

nodes = splitter.get_nodes_from_documents(documents)

if not nodes:
    raise RuntimeError("❌ Chunking لم يُنتج Nodes.")

print("✅ Documents:", len(documents))
print("✅ Nodes / Chunks:", len(nodes))

total_chars = sum(len(node.text) for node in nodes)

print("إجمالي الأحرف:", f"{total_chars:,}")
print("متوسط الأحرف لكل Chunk:", round(total_chars / len(nodes)))

print("\n--- مثال Chunk ---\n")
print(nodes[0].text[:1800])
print("\nMetadata:", nodes[0].metadata)


✅ Documents: 138
✅ Nodes / Chunks: 248
إجمالي الأحرف: 220,078
متوسط الأحرف لكل Chunk: 887

--- مثال Chunk ---

Damascus University logo

الجمهورية العربية السورية
جامعة دمشق
كلية الهندسة الزراعية
قسم الاقتصاد الزراعي

# دور سياسة التسويق الزراعي في تحسين كفاءة أداء أسواق الجملة لتجارة أهم منتجات الخضار والفاكهة في محافظة حماة

رسالة أُعدّت لنيل درجة الماجستير في الهندسة الزراعية
(اختصاص - اقتصاد زراعي)

**إعداد الطالبة**
**كنده إبراهيم زيدان**

**المشرف**
**الدكتور شباب ناصر**
أستاذ في قسم الاقتصاد الزراعي

**المشرف العلمي المشارك**
**الدكتور فادي مقدسي**
مدرس في قسم الاقتصاد الزراعي

العام الدراسي 1445هـ/2024م

Metadata: {'source': 'رسالة كندة ابراهيم زيدان.pdf', 'page': 1}



## 7. Cloud Embeddings عبر Jina

هذه الخلية تحتوي على batching وretry تلقائي لمعالجة حد الـtokens/minute.


In [ ]:

# ============================================================
# 7) Jina Cloud Embeddings مع Rate-Limit Handling
# ============================================================

import time
from llama_index.embeddings.jinaai import JinaEmbedding

JINA_API_KEY = os.environ["JINAAI_API_KEY"]

passage_embedder = JinaEmbedding(
    api_key=JINA_API_KEY,
    model=EMBED_MODEL_NAME,
    task="retrieval.passage",
    dimensions=EMBED_DIM,
    embed_batch_size=EMBED_BATCH_SIZE,
)

query_embedder = JinaEmbedding(
    api_key=JINA_API_KEY,
    model=EMBED_MODEL_NAME,
    task="retrieval.query",
    dimensions=EMBED_DIM,
    embed_batch_size=EMBED_BATCH_SIZE,
)


def embed_with_rate_limit(
    texts,
    embedder,
    batch_size=EMBED_BATCH_SIZE,
    wait_between_batches=WAIT_BETWEEN_BATCHES,
    retry_wait=RATE_LIMIT_RETRY_WAIT,
    max_retries=MAX_RETRIES,
):
    all_embeddings = []

    total_texts = len(texts)
    total_batches = (total_texts + batch_size - 1) // batch_size

    print(f"عدد النصوص: {total_texts}")
    print(f"عدد الـBatches: {total_batches}")
    print("-" * 60)

    for start in range(0, total_texts, batch_size):
        batch = texts[start:start + batch_size]

        batch_number = (start // batch_size) + 1
        retries = 0

        while True:
            try:
                print(
                    f"⏳ Embedding batch "
                    f"{batch_number}/{total_batches} "
                    f"({len(batch)} chunks)"
                )

                embeddings = embedder.get_text_embedding_batch(batch)

                all_embeddings.extend(embeddings)

                print(f"✅ Batch {batch_number} completed")
                break

            except RuntimeError as e:
                error_text = str(e).lower()

                if "rate limit" in error_text:
                    retries += 1

                    if retries > max_retries:
                        raise RuntimeError(
                            f"❌ تجاوزنا الحد الأقصى لإعادة المحاولة "
                            f"في Batch {batch_number}."
                        )

                    print(
                        f"⚠️ Jina Rate Limit — "
                        f"انتظار {retry_wait} ثانية "
                        f"(Retry {retries}/{max_retries})"
                    )

                    time.sleep(retry_wait)
                else:
                    raise

        if batch_number < total_batches:
            time.sleep(wait_between_batches)

    return all_embeddings


texts = [
    node.text
    for node in nodes
    if node.text.strip()
]

print("⏳ بدء Cloud Embedding عبر Jina...")

dense_vectors = embed_with_rate_limit(
    texts=texts,
    embedder=passage_embedder,
)

if not dense_vectors:
    raise RuntimeError("❌ Jina لم تُرجع embeddings.")

if len(dense_vectors) != len(nodes):
    raise RuntimeError(
        f"❌ عدد الـvectors ({len(dense_vectors)}) "
        f"لا يساوي عدد nodes ({len(nodes)})."
    )

actual_dim = len(dense_vectors[0])

if actual_dim != EMBED_DIM:
    raise RuntimeError(
        f"❌ Dimension الفعلي = {actual_dim} "
        f"بينما EMBED_DIM = {EMBED_DIM}"
    )

print()
print("=" * 60)
print("✅ تم إنشاء Embeddings بنجاح")
print("Vectors:", len(dense_vectors))
print("Dimension:", actual_dim)
print("=" * 60)


⏳ بدء Cloud Embedding عبر Jina...
عدد النصوص: 248
عدد الـBatches: 42
------------------------------------------------------------
⏳ Embedding batch 1/42 (6 chunks)
✅ Batch 1 completed
⏳ Embedding batch 2/42 (6 chunks)
✅ Batch 2 completed
⏳ Embedding batch 3/42 (6 chunks)
✅ Batch 3 completed
⏳ Embedding batch 4/42 (6 chunks)
✅ Batch 4 completed
⏳ Embedding batch 5/42 (6 chunks)
✅ Batch 5 completed
⏳ Embedding batch 6/42 (6 chunks)
✅ Batch 6 completed
⏳ Embedding batch 7/42 (6 chunks)
✅ Batch 7 completed
⏳ Embedding batch 8/42 (6 chunks)
✅ Batch 8 completed
⏳ Embedding batch 9/42 (6 chunks)
✅ Batch 9 completed
⏳ Embedding batch 10/42 (6 chunks)
✅ Batch 10 completed
⏳ Embedding batch 11/42 (6 chunks)
✅ Batch 11 completed
⏳ Embedding batch 12/42 (6 chunks)
✅ Batch 12 completed
⏳ Embedding batch 13/42 (6 chunks)
✅ Batch 13 completed
⏳ Embedding batch 14/42 (6 chunks)
✅ Batch 14 completed
⏳ Embedding batch 15/42 (6 chunks)
✅ Batch 15 completed
⏳ Embedding batch 16/42 (6 chunks)
✅ Batch 16 co


## 8. الاتصال بـQdrant Cloud وإنشاء Collection

> ملاحظة: لا نحذف الـCollection تلقائيًا إلا إذا اخترت ذلك في متغير `RESET_COLLECTION`.


In [ ]:

# ============================================================
# 8) Qdrant Cloud Connection + Collection Setup
# ============================================================

from qdrant_client import QdrantClient, models
from qdrant_client.http.exceptions import UnexpectedResponse

QDRANT_URL = os.environ["QDRANT_URL"]
QDRANT_API_KEY = os.environ["QDRANT_API_KEY"]

# للتجربة فقط:
# True  = حذف Collection القديمة وإعادة إنشائها
# False = الاحتفاظ بها
RESET_COLLECTION = True

qdrant = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=60,
)

print("🔍 اختبار الاتصال بـ Qdrant Cloud...")

collections_response = qdrant.get_collections()

existing_collections = [
    collection.name
    for collection in collections_response.collections
]

print("✅ الاتصال ناجح")
print(
    "Collections الحالية:",
    existing_collections if existing_collections else "لا يوجد"
)

exists = qdrant.collection_exists(COLLECTION_NAME)

if exists and RESET_COLLECTION:
    print(f"⚠️ حذف Collection القديمة: {COLLECTION_NAME}")

    qdrant.delete_collection(
        collection_name=COLLECTION_NAME
    )

    exists = False

    print("✅ تم الحذف")

if not exists:
    print("🚀 إنشاء Collection جديدة...")

    qdrant.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            "dense_vector": models.VectorParams(
                size=EMBED_DIM,
                distance=models.Distance.COSINE,
            )
        },
        sparse_vectors_config={
            "bm25_sparse_vector":
                models.SparseVectorParams(
                    modifier=models.Modifier.IDF
                )
        },
    )

    print("✅ تم إنشاء Collection")

else:
    print("✅ Collection موجودة ولن يتم حذفها:", COLLECTION_NAME)

collection_info = qdrant.get_collection(
    collection_name=COLLECTION_NAME
)

print()
print("=" * 60)
print("Collection:", COLLECTION_NAME)
print("Dense dimension:", EMBED_DIM)
print("Dense distance: COSINE")
print("Sparse: Qdrant BM25 + IDF")
print("=" * 60)


🔍 اختبار الاتصال بـ Qdrant Cloud...
✅ الاتصال ناجح
Collections الحالية: ['cloud_rag_demo']
⚠️ حذف Collection القديمة: cloud_rag_demo
✅ تم الحذف
🚀 إنشاء Collection جديدة...
✅ تم إنشاء Collection

Collection: cloud_rag_demo
Dense dimension: 1024
Dense distance: COSINE
Sparse: Qdrant BM25 + IDF



## 9. رفع الـChunks والـVectors إلى Qdrant

هذه هي المرحلة التي كانت مفقودة في النسخة السابقة.


In [ ]:

# ============================================================
# 9) Upload Nodes to Qdrant
# ============================================================

import uuid

if len(nodes) != len(dense_vectors):
    raise RuntimeError(
        f"❌ nodes={len(nodes)} بينما vectors={len(dense_vectors)}"
    )

points = []

for index, (node, dense_vector) in enumerate(
    zip(nodes, dense_vectors),
    start=1,
):
    text = node.text.strip()

    if not text:
        continue

    payload = {
        "text": text,
        "source": node.metadata.get("source"),
        "page": node.metadata.get("page"),
        "node_id": node.node_id,
        "chunk_index": index,
    }

    points.append(
        models.PointStruct(
            id=str(uuid.uuid4()),
            payload=payload,
            vector={
                "dense_vector": dense_vector,
                "bm25_sparse_vector": models.Document(
                    text=text,
                    model="Qdrant/bm25",
                    options={
                        "language": "none",
                        "tokenizer": "multilingual",
                    },
                ),
            },
        )
    )

if not points:
    raise RuntimeError("❌ لا توجد points جاهزة للرفع.")

print(f"⏳ رفع {len(points)} Points إلى Qdrant...")

qdrant.upload_points(
    collection_name=COLLECTION_NAME,
    points=points,
    batch_size=16,
    max_retries=3,
)

count_result = qdrant.count(
    collection_name=COLLECTION_NAME,
    exact=True,
)

print()
print("=" * 60)
print("✅ تم رفع البيانات إلى Qdrant")
print("Points المرسلة:", len(points))
print("Points داخل Qdrant:", count_result.count)
print("=" * 60)

if count_result.count == 0:
    raise RuntimeError("❌ Collection فارغة بعد عملية الرفع.")


⏳ رفع 248 Points إلى Qdrant...

✅ تم رفع البيانات إلى Qdrant
Points المرسلة: 248
Points داخل Qdrant: 248


## 10. اختبار مباشر لبيانات Qdrant قبل بناء الـRAG

In [ ]:

# ============================================================
# 10) فحص محتوى Qdrant
# ============================================================

scroll_result, next_page = qdrant.scroll(
    collection_name=COLLECTION_NAME,
    limit=min(5, len(points)),
    with_payload=True,
    with_vectors=False,
)

print("عدد النتائج المعروضة:", len(scroll_result))

for i, point in enumerate(scroll_result, start=1):
    print("=" * 80)
    print("Point:", i)
    print("Page:", point.payload.get("page"))
    print("Source:", point.payload.get("source"))
    print(point.payload.get("text", "")[:1000])


عدد النتائج المعروضة: 5
Point: 1
Page: 107
Source: رسالة كندة ابراهيم زيدان.pdf
الشكل (34): نسبة صافي الهامش التسويقي للكيلوغرام الواحد من الفاكهة للوسطاء في العملية التسويقية لمتوسط عامي 2021 -2022.

المصدر: نتائج تحليل بيانات المسح الميداني لعام 2023، محافظة حماه.
Point: 2
Page: 111
Source: رسالة كندة ابراهيم زيدان.pdf
الجدول رقم (34): نتائج تطبيق نموذج الانحدار الخطي المتعدد

| Sig   | Beta   | T      | B      | البيان                                         |
| ----- | ------ | ------ | ------ | ---------------------------------------------- |
| 0.000 |        | 29.297 | 53.766 | (Constant) الثابت                              |
| 0.001 | -0.472 | -3.409 | -3.273 | هل شعرت بزيادة عدد الوسطاء X₁                  |
| 0.002 | -1.138 | -3.134 | -8.650 | هل شعرت بتحسن في بنية أسواق الجملة X₂          |
| 0.000 | 0.285  | 3.812  | 1.629  | هل شعرت بتخفيض رسوم عمولات الأسواق المركزية X₃ |
| 0.203 | 0.436  | 1.283  | 3.447  | هل تتلقى معلومات عن الأسعار قبل التسويق X₄     |
| 0.000 | 0.639 


## 11. Hybrid Retrieval

- Dense Retrieval عبر Jina query embedding.
- Sparse Retrieval عبر Qdrant BM25.
- دمج النتائج باستخدام RRF.


In [ ]:

# ============================================================
# 11) Hybrid Retrieval
# ============================================================

def hybrid_retrieve(question: str, top_k: int = RRF_TOP_K):
    if not question or not question.strip():
        raise ValueError("السؤال فارغ.")

    query_vector = query_embedder.get_query_embedding(
        question.strip()
    )

    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            models.Prefetch(
                query=query_vector,
                using="dense_vector",
                limit=DENSE_CANDIDATES,
            ),
            models.Prefetch(
                query=models.Document(
                    text=question.strip(),
                    model="Qdrant/bm25",
                    options={
                        "language": "none",
                        "tokenizer": "multilingual",
                    },
                ),
                using="bm25_sparse_vector",
                limit=SPARSE_CANDIDATES,
            ),
        ],
        query=models.FusionQuery(
            fusion=models.Fusion.RRF
        ),
        limit=top_k,
        with_payload=True,
    )

    return results.points


## 12. اختبار Retrieval قبل الـReranker والـLLM

In [ ]:

# ============================================================
# 12) Retrieval Debug Helper
# ============================================================

def debug_retrieval(question: str, top_k: int = 5):
    results = hybrid_retrieve(
        question,
        top_k=top_k,
    )

    print("السؤال:", question)
    print("عدد النتائج:", len(results))
    print()

    for i, item in enumerate(results, start=1):
        payload = item.payload or {}

        print("=" * 80)
        print(f"Result #{i}")
        print("RRF score:", item.score)
        print("Page:", payload.get("page"))
        print("Source:", payload.get("source"))
        print()
        print(payload.get("text", "")[:1500])

    return results


## 13. Cloud Reranking عبر Jina + LlamaIndex

In [ ]:

# ============================================================
# 13) Jina Cloud Reranker
# ============================================================

from llama_index.postprocessor.jinaai_rerank import JinaRerank
from llama_index.core.schema import TextNode, NodeWithScore, QueryBundle

reranker = JinaRerank(
    api_key=JINA_API_KEY,
    model=RERANK_MODEL_NAME,
    top_n=RERANK_TOP_N,
)


def rerank_results(question: str, qdrant_results):
    if not qdrant_results:
        return []

    llama_nodes = []

    for result in qdrant_results:
        payload = result.payload or {}

        node = TextNode(
            text=payload.get("text", ""),
            metadata={
                "source": payload.get("source"),
                "page": payload.get("page"),
                "qdrant_id": str(result.id),
                "chunk_index": payload.get("chunk_index"),
            },
        )

        llama_nodes.append(
            NodeWithScore(
                node=node,
                score=float(result.score or 0),
            )
        )

    reranked = reranker.postprocess_nodes(
        llama_nodes,
        query_bundle=QueryBundle(
            query_str=question
        ),
    )

    return reranked


## 14. Hugging Face LLM Client

In [ ]:

# ============================================================
# 14) Hugging Face LLM
# ============================================================

from openai import OpenAI

llm_client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"],
)

print("✅ Hugging Face client جاهز")


✅ Hugging Face client جاهز


## 15. Prompt وبناء الـContext

In [ ]:

# ============================================================
# 15) Prompt + Context
# ============================================================

SYSTEM_PROMPT = """
أنت مساعد عربي يعمل ضمن نظام RAG للإجابة عن الأسئلة اعتماداً على الوثائق المسترجعة.

التزم بالقواعد التالية بدقة:

1. استخدم السياق المرفق فقط.
2. لا تعتمد على معلومات خارجية أو على معرفتك العامة.
3. إذا كان السياق يحتوي على الجواب، أجب مباشرة وبوضوح.
4. إذا لم يحتوِ السياق على معلومات كافية فعلاً، قل:
   لا توجد معلومات كافية في الوثائق للإجابة بدقة.
5. لا تخترع معلومات أو مصادر.
6. أشر إلى المصدر داخل الإجابة بصيغة [المصدر 1] أو [المصدر 2].
7. أجب باللغة العربية ما لم يطلب المستخدم لغة أخرى.
8. لا ترفض الإجابة فقط لأن عدد المصادر قليل؛ مصدر واحد كافٍ إذا كان يحتوي على الجواب.
"""


def build_context(reranked_nodes):
    blocks = []

    for i, item in enumerate(
        reranked_nodes,
        start=1,
    ):
        page = item.node.metadata.get(
            "page",
            "?"
        )

        source = item.node.metadata.get(
            "source",
            "?"
        )

        blocks.append(
            f"[المصدر {i}]\n"
            f"الملف: {source}\n"
            f"الصفحة: {page}\n\n"
            f"{item.node.text}"
        )

    return "\n\n".join(blocks)


## 16. تابع RAG النهائي

In [ ]:

# ============================================================
# 16) ask_rag()
# ============================================================

def ask_rag(
    question: str,
    verbose: bool = False,
):
    question = (question or "").strip()

    if not question:
        return {
            "question": "",
            "answer": "الرجاء إدخال سؤال.",
            "sources": [],
            "debug": {},
        }

    # 1) Hybrid Retrieval
    candidates = hybrid_retrieve(question)

    if not candidates:
        return {
            "question": question,
            "answer": (
                "لم يتم العثور على أي مقاطع مسترجعة من Qdrant. "
                "تحقق من أن الـCollection تحتوي Points."
            ),
            "sources": [],
            "debug": {
                "retrieved_count": 0,
                "reranked_count": 0,
            },
        }

    # 2) Cloud Reranking
    reranked_nodes = rerank_results(
        question,
        candidates,
    )

    if not reranked_nodes:
        return {
            "question": question,
            "answer": (
                "تم الاسترجاع من Qdrant، "
                "لكن الـReranker لم يُرجع نتائج."
            ),
            "sources": [],
            "debug": {
                "retrieved_count": len(candidates),
                "reranked_count": 0,
            },
        }

    # 3) Context
    context = build_context(
        reranked_nodes
    )

    user_prompt = f"""
السؤال:
{question}

السياق المسترجع من الوثائق:
-----------------------------
{context}
-----------------------------

أجب عن السؤال بالاعتماد على السياق أعلاه فقط.
إذا كان الجواب موجوداً في أحد المصادر، أجب به حتى لو كان مصدرًا واحدًا فقط.
"""

    # 4) LLM generation
    completion = llm_client.chat.completions.create(
        model=LLM_MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
        temperature=0.2,
        max_tokens=2048,
        extra_body={
            "chat_template_kwargs": {
                "enable_thinking": False
            }
        },
    )

    message = completion.choices[0].message
    answer = (message.content or "").strip()

    if not answer:
        answer = (
            "تم استرجاع المصادر بنجاح، "
            "لكن نموذج اللغة أعاد إجابة فارغة."
        )

    sources = []

    for i, item in enumerate(
        reranked_nodes,
        start=1,
    ):
        sources.append(
            {
                "source_number": i,
                "file": item.node.metadata.get(
                    "source"
                ),
                "page": item.node.metadata.get(
                    "page"
                ),
                "reranker_score": item.score,
                "text_preview": item.node.text[:350],
            }
        )

    result = {
        "question": question,
        "answer": answer,
        "sources": sources,
        "debug": {
            "retrieved_count": len(candidates),
            "reranked_count": len(reranked_nodes),
        },
    }

    if verbose:
        print("# الإجابة")
        print(answer)
        print()
        print("# المصادر")

        for source in sources:
            print(
                f"[المصدر {source['source_number']}] "
                f"{source['file']} "
                f"- الصفحة {source['page']} "
                f"- score={source['reranker_score']}"
            )

    return result



# 17. واجهة سؤال بمربع نص وزر

اكتب السؤال في مربع النص واضغط **اسأل النظام**.

يمكنك طرح عدة أسئلة متتالية بدون إعادة أي خلية سابقة.


In [ ]:

# ============================================================
# 17) Interactive Question Box
# ============================================================

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

question_box = widgets.Textarea(
    value="",
    placeholder="اكتب سؤالك عن الوثيقة هنا...",
    description="السؤال:",
    layout=widgets.Layout(
        width="100%",
        height="100px",
    ),
    style={
        "description_width": "70px"
    },
)

ask_button = widgets.Button(
    description="اسأل النظام",
    button_style="primary",
    icon="search",
    layout=widgets.Layout(
        width="180px"
    ),
)

debug_checkbox = widgets.Checkbox(
    value=False,
    description="إظهار تفاصيل الاسترجاع",
)

output_box = widgets.Output()


def on_ask_clicked(_):
    question = question_box.value.strip()

    with output_box:
        clear_output(wait=True)

        if not question:
            display(
                Markdown(
                    "⚠️ **اكتب سؤالًا أولًا.**"
                )
            )
            return

        print("⏳ جاري البحث وتوليد الإجابة...")

        try:
            result = ask_rag(
                question,
                verbose=False,
            )

            clear_output(wait=True)

            display(
                Markdown(
                    "## الإجابة\n\n"
                    + result["answer"]
                )
            )

            if result["sources"]:
                display(
                    Markdown(
                        "\n## المصادر المسترجعة"
                    )
                )

                for source in result["sources"]:
                    score = source["reranker_score"]

                    display(
                        Markdown(
                            f"**[المصدر {source['source_number']}]**  \n"
                            f"- الملف: `{source['file']}`  \n"
                            f"- الصفحة: **{source['page']}**  \n"
                            f"- Reranker score: `{score}`"
                        )
                    )

            if debug_checkbox.value:
                debug = result.get(
                    "debug",
                    {}
                )

                display(
                    Markdown(
                        "\n## Debug"
                    )
                )

                print(
                    "Retrieved candidates:",
                    debug.get(
                        "retrieved_count"
                    )
                )

                print(
                    "Reranked nodes:",
                    debug.get(
                        "reranked_count"
                    )
                )

                if result["sources"]:
                    print(
                        "\nTop retrieved text previews:\n"
                    )

                    for source in result["sources"]:
                        print(
                            "=" * 70
                        )
                        print(
                            f"Source {source['source_number']} "
                            f"- Page {source['page']}"
                        )
                        print(
                            source[
                                "text_preview"
                            ]
                        )

        except Exception as e:
            clear_output(wait=True)

            display(
                Markdown(
                    "## ❌ حدث خطأ"
                )
            )

            print(
                type(e).__name__,
                str(e),
            )


ask_button.on_click(
    on_ask_clicked
)

display(
    widgets.VBox(
        [
            question_box,
            widgets.HBox(
                [
                    ask_button,
                    debug_checkbox,
                ]
            ),
            output_box,
        ]
    )
)



## 18. اختبار تشخيصي اختياري

إذا شعرت أن جواب النظام غير صحيح، استخدم الخلية التالية لفحص الاسترجاع فقط بدون الـLLM.


In [ ]:

# مثال:
# debug_retrieval("شو اسم وزير الزراعة؟", top_k=5)



# Flow النهائي

```text
Upload PDF
   ↓
LlamaParse Cloud
   ↓
LlamaIndex Chunking
   ↓
Jina Embeddings API
   ↓
Create / Connect Qdrant Collection
   ↓
UPLOAD POINTS ← خطوة أساسية
   ↓
Verify Qdrant Count
   ↓
User enters question in UI
   ↓
Jina Query Embedding
   ↓
Dense Retrieval + Qdrant BM25
   ↓
RRF Hybrid Fusion
   ↓
Jina Cloud Reranker
   ↓
Top-N context
   ↓
Hugging Face LLM
   ↓
Arabic Answer + Citations
```

## ملاحظة للمشروع النهائي

في FastAPI الحقيقي:
- مرحلة ingestion يجب أن تعمل كـbackground job.
- لا تحذف الـCollection عند تشغيل الخدمة.
- استخدم `document_id` و`user_id` داخل metadata والفلاتر.
- لا تعِد embedding لوثيقة سبق معالجتها إلا عند reprocess.
